# 03 · Data Fusion

Fuse all data sources to the zip × hurricane grain. Apply HUD crosswalk for tract-to-zip, geodesic distances via `pyproj.Geod`, and flood overlay in EPSG:5070.

In [11]:
import sys, os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 60)
from config import (
    DATA_PATHS, HURRICANE_META, STATES_IN_SCOPE,
    TARGET_COL, TARGET_CLASS_COL, FEATURE_GROUPS,
    RANDOM_STATE, SEVERITY_BINS, SEVERITY_LABELS,
)
RAW = DATA_PATHS['raw']; INTERIM = DATA_PATHS['interim']
PROC = DATA_PATHS['processed']; MODELS = DATA_PATHS['models']
OUT = DATA_PATHS['outputs']


In [12]:
import geopandas as gpd
from src.data_fusion import (
    merge_housing_assistance, tract_to_zip, snap_retailers_per_zip,
    build_track_linestring, compute_distance_to_track, pct_in_floodplain,
)

### Housing Assistance (Owners + Renters summed, per hurricane)

In [13]:
ha_frames = []
for h in HURRICANE_META:
    dn = h['disaster_number']
    o = pd.read_csv(RAW / f'fema_housing_owners_{dn}.csv')
    r = pd.read_csv(RAW / f'fema_housing_renters_{dn}.csv')
    ha_frames.append(merge_housing_assistance(o, r))
ha = pd.concat(ha_frames, ignore_index=True)
print('rows (zip × hurricane):', len(ha))

rows (zip × hurricane): 30161


### ACS demographics

In [14]:
acs = pd.read_csv(RAW / 'census_acs5_zcta.csv', dtype={'zip_code': str})
acs['zip_code'] = acs['zip_code'].str.zfill(5)
from src.feature_engineering import derive_demographic_shares
acs = derive_demographic_shares(acs)
print(acs.shape)

(33774, 27)


### CDC SVI — replace -999, then tract→zip via HUD

In [15]:
svi = pd.read_csv(RAW / 'cdc_svi_2022.csv', low_memory=False)
svi = svi.replace(-999, np.nan)
cw = pd.read_excel(RAW / 'hud_tract_zip.xlsx')
svi_keep = ['FIPS','RPL_THEME1','RPL_THEME2','RPL_THEME3','RPL_THEME4','RPL_THEMES']
svi_zip = tract_to_zip(svi[svi_keep], cw, tract_col='FIPS',
                       value_cols=svi_keep[1:])
svi_zip = svi_zip.rename(columns={
    'RPL_THEME1':'svi_socioeconomic','RPL_THEME2':'svi_household_comp',
    'RPL_THEME3':'svi_minority_lang','RPL_THEME4':'svi_housing_transport',
    'RPL_THEMES':'svi_overall'})
print(svi_zip.shape); svi_zip.head()

(39180, 6)


,zip_code,svi_socioeconomic,svi_household_comp,svi_minority_lang,svi_housing_transport,svi_overall
0,00501,0.000000,0.000000,0.000000,0.000000,0.000000
1,01001,0.365519,0.547502,0.239272,0.566116,0.438860
2,01002,0.417859,0.227482,0.443690,0.653897,0.430972
3,01003,0.482757,0.146292,0.455233,0.693876,0.446146
4,01004,0.622700,0.209500,0.411700,0.848000,0.620000


### USDA Food Access Atlas (tract → zip)

In [16]:
fa = pd.read_excel(RAW / 'food_access_atlas.xlsx', sheet_name='Food Access Research Atlas')
fa_keep = ['CensusTract','LILATracts_1And10','lalowi1','lahunv1share','TractSNAP']
fa_zip = tract_to_zip(fa[fa_keep], cw, tract_col='CensusTract', value_cols=fa_keep[1:])
fa_zip['food_desert_flag'] = (fa_zip['LILATracts_1And10'] >= 0.5).astype(int)
fa_zip = fa_zip.rename(columns={'TractSNAP':'snap_participation_pct'})
print(fa_zip.shape)

(35585, 6)


### SNAP retailers — spatial join to ZCTA

In [17]:
import glob
snap_csv = next((p for p in (RAW / 'snap_retailers').rglob('*.csv')), None)
snap = pd.read_csv(snap_csv)
zcta_path = next((RAW / 'zcta').rglob('*.shp'))
zcta_gdf = gpd.read_file(zcta_path)
snap_zip = snap_retailers_per_zip(snap, zcta_gdf)
print(snap_zip.shape); snap_zip.head()

(33791, 3)


,zip_code,snap_retailer_count,dist_nearest_supermarket_mi
0,00601,0,116.341963
1,00602,0,147.499727
2,00603,0,142.887845
3,00606,0,128.328099
4,00610,0,141.538819


### IBTrACS distance-to-track per hurricane (geodesic)

In [18]:
ibt = pd.read_csv(RAW / 'ibtracs_na.csv', low_memory=False, skiprows=[1])
dist_rows = []
for h in HURRICANE_META:
    track = build_track_linestring(ibt, h['name'], h['year'])
    # restrict zip set to states affected by this hurricane for speed
    zcta_sub = zcta_gdf  # for full rigor, filter by state
    d = compute_distance_to_track(zcta_sub, track)
    d['disaster_number'] = h['disaster_number']
    dist_rows.append(d)
dist = pd.concat(dist_rows, ignore_index=True)
print(dist.shape)

c:\Users\chaitanya\Documents\ML Project\src\data_fusion.py:154: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cents = zcta_gdf.to_crs(CRS_LATLON).geometry.centroid
dist_to_track: 100%|██████████| 33791/33791 [00:00<00:00, 42961.24it/s]
c:\Users\chaitanya\Documents\ML Project\src\data_fusion.py:154: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cents = zcta_gdf.to_crs(CRS_LATLON).geometry.centroid
dist_to_track: 100%|██████████| 33791/33791 [00:00<00:00, 56830.06it/s]
c:\Users\chaitanya\Documents\ML Project\src\data_fusion.py:154: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation

(236537, 3)


### Flood overlay (EPSG:5070, county-by-county)

In [19]:
flood_rows = []
for st_fips in ['48','22','12','37','45','13','01','28']:
    p = RAW / f'nfhl_sfha_{st_fips}.geojson'
    if not p.exists(): continue
    nfhl = gpd.read_file(p)
    flood_rows.append(pct_in_floodplain(zcta_gdf, nfhl))
flood = pd.concat(flood_rows, ignore_index=True) if flood_rows else pd.DataFrame({'zip_code':[],'pct_in_100yr_floodplain':[]})
flood = flood.groupby('zip_code')['pct_in_100yr_floodplain'].max().reset_index()
print(flood.shape)

flood overlay by county: 100%|██████████| 1/1 [00:12<00:00, 12.79s/it]

(95, 2)


### Merge everything

In [20]:
fused = (ha
    .merge(acs, on='zip_code', how='left')
    .merge(svi_zip, on='zip_code', how='left')
    .merge(fa_zip, on='zip_code', how='left')
    .merge(snap_zip, on='zip_code', how='left')
    .merge(dist, on=['zip_code','disaster_number'], how='left')
    .merge(flood, on='zip_code', how='left')
)
fused['snap_retailers_per_1k'] = fused['snap_retailer_count'] / (fused['population'].replace(0, np.nan) / 1000)
print('fused shape:', fused.shape)
fused.to_csv(INTERIM / 'fused_zip_hurricane.csv', index=False)
fused.head()

fused shape: (30161, 55)


,disaster_number,zip_code,state,county,total_inspected,total_major_substantial,total_no_damage,total_minor,total_moderate,total_approved_dollars,validRegistrations,repairReplaceAmount,rentalAmount,otherNeedsAmount,NAME,population,median_income,poverty_count,renters,male_65_66,male_67_69,male_70_74,male_75_79,male_80_84,male_85_over,female_65_66,female_67_69,female_70_74,female_75_79,female_80_84,female_85_over,white_alone,no_vehicle_households,mobile_homes,pct_poverty,pct_renters,pct_elderly_65plus,pct_minority,pct_no_vehicle,pct_mobile_homes,svi_socioeconomic,svi_household_comp,svi_minority_lang,svi_housing_transport,svi_overall,LILATracts_1And10,lalowi1,lahunv1share,snap_participation_pct,food_desert_flag,snap_retailer_count,dist_nearest_supermarket_mi,distance_to_track_km,pct_in_100yr_floodplain,snap_retailers_per_1k
0,4332,00000,TX,Harris (County),0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4332,00000,TX,Waller (County),1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4332,00961,TX,Bastrop (County),1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,ZCTA5 00961,30842.0,34346.0,8719.0,3729.0,265.0,514.0,770.0,560.0,403.0,390.0,476.0,632.0,1069.0,849.0,732.0,584.0,15784.0,1458.0,40.0,28.269892,12.090656,23.487452,48.823034,4.727320,0.129693,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,79.342023,534.117596,NaN,0.000000
3,4332,11111,TX,Jefferson (County),0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4332,28326,TX,Fort Bend (County),1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,ZCTA5 28326,24998.0,72545.0,4011.0,3176.0,180.0,127.0,256.0,118.0,45.0,93.0,285.0,125.0,290.0,99.0,139.0,78.0,15899.0,234.0,1719.0,16.045284,12.705016,7.340587,36.398912,0.936075,6.876550,0.61382,0.664419,0.575892,0.317428,0.553991,0.0,2056.327092,6.546646,263.59731,0.0,30.0,0.000000,590.593461,NaN,1.200096
